# 00 — IOI baseline

Load GPT-2 small, confirm the **Indirect Object Identification** behavior is there before we start taking the circuit apart.

IOI in one line: given *"When John and Mary went to the store, John gave a drink to"*, the model should continue with **" Mary"** (the indirect object, mentioned once) and **not** **" John"** (the subject, mentioned twice). The metric is the **logit difference** `logit(' Mary') - logit(' John')` — positive means the behavior is present.

Reference: Wang et al., *Interpretability in the Wild* (2022).

In [ ]:
import torch
from transformer_lens import HookedTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {device}", f"({torch.cuda.get_device_name(0)})" if device == "cuda" else "")

In [ ]:
# gpt2 = GPT-2 small (124M). transformer-lens folds LayerNorm + centers the
# weights by default, which makes the residual stream cleaner to interpret.
model = HookedTransformer.from_pretrained("gpt2", device=device)
model.eval()
print(f"layers={model.cfg.n_layers}  heads={model.cfg.n_heads}  d_model={model.cfg.d_model}")

## The canonical prompt

`S` = subject (John), `IO` = indirect object (Mary). We read the next-token logits at the final position and compare the two name tokens.

In [ ]:
prompt = "When John and Mary went to the store, John gave a drink to"

io_tok = model.to_single_token(" Mary")  # indirect object — the correct answer
s_tok  = model.to_single_token(" John")  # subject — the distractor

logits = model(prompt)[0, -1]            # [d_vocab] logits for the next token
logit_diff = (logits[io_tok] - logits[s_tok]).item()

print(f"top predicted token: {model.tokenizer.decode(logits.argmax().item())!r}")
print(f"logit_diff (IO Mary - S John): {logit_diff:+.3f}")
print("IOI behavior present" if logit_diff > 0 else "NO IOI (unexpected)")

Expect the top token to be `' Mary'` and the logit diff to be **≈ +3.2**. That positive gap is the thing the IOI circuit produces; the rest of the lab is about explaining *how*.

## A small batch + per-prompt diffs

Single prompts are noisy. The real IOI work uses balanced templates (swapping which name is S vs IO, and the two sentence orders) so the metric averages over surface form. A first taste — flip the names and confirm the behavior tracks the *role*, not the string `Mary`.

In [ ]:
def ioi_logit_diff(prompt, io_name, s_name):
    logits = model(prompt)[0, -1]
    io = model.to_single_token(" " + io_name)
    s  = model.to_single_token(" " + s_name)
    return (logits[io] - logits[s]).item()

cases = [
    ("When John and Mary went to the store, John gave a drink to", "Mary", "John"),
    ("When Mary and John went to the store, Mary gave a drink to", "John", "Mary"),
    ("When Alice and Bob went to the park, Bob gave the ball to",  "Alice", "Bob"),
    ("When Bob and Alice went to the park, Alice gave the ball to", "Bob", "Alice"),
]
for prompt, io, s in cases:
    print(f"{ioi_logit_diff(prompt, io, s):+.3f}   IO={io:<6} S={s}")

All four should be positive — the circuit identifies the *indirect object role*, regardless of which name fills it.

## Cache the activations

`run_with_cache` stashes every internal activation. This is the raw material for the next notebook (activation patching to localize which heads/positions carry the IOI signal).

In [ ]:
logits, cache = model.run_with_cache(prompt)
# A few of the hook points we'll reach for next: per-head attention patterns,
# per-layer residual stream, attention head outputs (z).
print("resid_post L0 :", tuple(cache["resid_post", 0].shape))   # [batch, pos, d_model]
print("attn pattern L0:", tuple(cache["pattern", 0].shape))      # [batch, head, q, k]
print("head out z L9  :", tuple(cache["z", 9].shape))            # [batch, pos, head, d_head]

Next: `01_activation_patching.ipynb` — corrupt the prompt (swap a name), patch clean activations back in head-by-head, and watch which ones restore the logit diff. That's how the name-mover heads get localized.